# Supplementary analysis: default trees across canonical strata

Descriptive assessment of default decision-tree performance, generalization gaps, and structural properties across fixed difficulty strata. Each dataset is a separate classification task; observations are not pooled across datasets.


## Reproducibility and execution

This supplement consists of three notebooks in `experiments/`. Run each notebook
from top to bottom with a fresh Python kernel, in this order:

1. `New_Characterization_Experiments.ipynb`: canonical baseline evaluations,
   difficulty scores, score ablation and baseline-dependence experiments.
2. `Default_Tree_Canonical_Strata.ipynb`: default-tree performance and structure
   conditional on the fixed canonical strata.
3. `Dataset_Characterization.ipynb`: descriptive characterization, internal
   consistency checks and sensitivity analyses of saved canonical results.

**Inputs.** Supply the 20 preprocessed CSV files in `data/` at the supplement
root. Exact filenames, target columns and identifier exclusions are defined in
the data-loading cells of the first notebook. Raw data and generated results
are not included in these three source files. Reusing saved results requires
the original CSV bytes, filtered data, software versions and partition hashes;
the notebooks stop on incompatible inputs. Dataset acquisition and preprocessing
must be supplied separately with the publication.

**Environment.** The saved reference uses Python 3.11.5, NumPy 1.23.5,
pandas 2.3.3, SciPy 1.15.3 and scikit-learn 1.3.0. The recorded auxiliary packages are Matplotlib 3.10.8, joblib 1.5.3
and threadpoolctl 3.5.0. IPython and Jupyter are also required. The complete recorded
environment is exported in `results/new_characterization_experiments/metadata.json`.
Saved partition hashes are platform-sensitive; matching package versions alone
does not guarantee compatibility with results created on another platform.

**Protocol.** Use repeated stratified 5-fold cross-validation with 3 repeats,
seed 42, macro-F1 (`zero_division=0`) and sample standard deviation (`ddof=1`).
Preprocessing is fitted within each training fold. Dummy classifiers are
reference baselines and do not contribute to difficulty scores. The canonical
score is 0.70 times normalized predictive error plus 0.30 times normalized
within-model CV variability. Eligibility and exact-tie rules remain defined
in the analysis code; the saved pool contains 17 eligible datasets in strata
of 6, 6 and 5. No hyperparameter search is performed.

**Outputs.** The first two notebooks write under `results/`; the third writes
under `experiments/canonical_alignment/outputs/`. Alternate-seed evaluations
require compatible saved caches or explicit activation of `ALLOW_SEED_FITS`
in the third notebook. Existing source results are checked for consistency.

**Source organization.** Supporting functions are included as readable library
cells at the end of these notebooks. The setup cell loads only the tagged
library definitions into isolated Python namespaces; it does not execute the
other notebooks' analyses or create Python source files. Library cells are
skipped during ordinary top-to-bottom execution because setup already loaded them.

**Interpretation.** Dataset-level summaries are descriptive. Validation using
the same baselines that construct the strata is internal consistency evidence,
not independent predictive validation. Exploratory p-values are unadjusted.
Stored outputs and execution counters have been cleared for this source edition;
run the notebooks to generate tables and figures. The recorded methodology is
preserved; no new scientific results are asserted by this editorial cleanup.

In [ ]:
# Load the documented helper cells from this three-notebook supplement.
from pathlib import Path
import json
import sys
import types

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'experiments' / 'New_Characterization_Experiments.ipynb').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run from the supplement root or its experiments directory.')

def load_supplement_library(module_name, notebook_name, tag, relative_module_path):
    """Load only the explicitly tagged library cell; never run analysis cells."""
    path = PROJECT_ROOT / 'experiments' / notebook_name
    notebook = json.loads(path.read_text(encoding='utf-8'))
    cells = [c for c in notebook['cells']
             if c['cell_type'] == 'code' and tag in c.get('metadata', {}).get('tags', [])]
    if len(cells) != 1:
        raise ValueError(f'Expected one {tag} library cell in {path.name}.')
    for package in ['experiments', 'experiments.canonical_alignment']:
        if package not in sys.modules:
            namespace = types.ModuleType(package)
            namespace.__path__ = []
            sys.modules[package] = namespace
    module = types.ModuleType(module_name)
    module.__file__ = str(PROJECT_ROOT / relative_module_path)
    sys.modules[module_name] = module
    try:
        exec(compile(''.join(cells[0]['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(module_name, None)
        raise
    return module

load_supplement_library(
    'experiments.characterization_persistence', 'New_Characterization_Experiments.ipynb',
    'supplement-persistence', 'experiments/characterization_persistence.py',
)
load_supplement_library(
    'experiments.default_tree_strata', 'Default_Tree_Canonical_Strata.ipynb',
    'supplement-tree', 'experiments/default_tree_strata.py',
)


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments' / 'New_Characterization_Experiments.ipynb').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from experiments.default_tree_strata import (
    prepare, fit_folds, summarize, plot_comparisons, plot_relations,
    plot_occupancy, interpretation, paper_text,
)
context = prepare(PROJECT_ROOT)
display(context['canonical'][['dataset', 'difficulty_group', 'difficulty_score', 'position']].sort_values('position'))
display(context['exclusions'])
print(json.dumps(context['metadata']['versions'], indent=2))
print(json.dumps(context['metadata']['tree_parameters'], indent=2))


## Eligibility, source reuse and provenance

The loader, target/ID configuration, baseline preprocessor, eligibility aggregation and canonical grouping functions are read directly from the original notebook without running its fitting or export stages. Raw-file and filtered-data hashes, saved code identities, versions, all four required baseline evaluations, the 17-member pool and labels are checked before fitting. The original notebook and its results are not modified. Historical eligible labels are also compared when available; score differences are reported separately from label agreement.

Canonical labels are read from the saved canonical panel and checked against the original profile and saved baseline results. **No new tree result enters label assignment.** The score combines 0.70 times min-max normalized predictive error with 0.30 times min-max normalized within-model CV standard deviation, across the four non-dummy baselines. Its existing deterministic ranking is split into thirds (6/6/5). Abalone, arrhythmia and bridges fail the canonical minimum-class-count criterion; exclusions are displayed above. Discrepancies stop execution before downstream analysis.

## Estimator, preprocessing and validation

Only `DecisionTreeClassifier(random_state=42)` is instantiated; the effective defaults and scikit-learn version are printed and saved in `metadata.json`. Numerical features use median imputation. Categorical features use most-frequent imputation and dense one-hot encoding with unknown categories ignored, matching the baseline. Each fresh pipeline fits its imputer and encoder on training rows only. Target encoding and missing-target filtering follow the original loader.

The **saved train/test indices** are reused for 5 folds × 3 repetitions, seed 42; they are also checked against the original repeated stratified protocol. Indices refer to the filtered dataset with reset indices. There are 15 independently fitted instances per dataset, 255 total. Macro-F1 uses `average='macro', zero_division=0`. Every validation F1, depth, node count and leaf count must reproduce the saved default baseline fold. Retraining is necessary because canonical artifacts did not retain fitted estimators, training F1 or leaf occupancies.

Difficulty stratification concerns **selection of datasets**. Class stratification concerns the distribution of class labels **within CV folds**; they are different operations.


In [ ]:
fold_metrics, dataset_metadata = fit_folds(context)
display(fold_metrics.head())
print(f'Executed {len(fold_metrics)} folds; captured warnings in {fold_metrics.warnings.ne("").sum()} folds.')
warning_audit = fold_metrics.loc[fold_metrics.warnings.ne(''), ['dataset', 'repeat', 'fold', 'warnings']]
warning_audit.to_csv(context['out'] / 'tables/preprocessing_warnings.csv', index=False)
display(warning_audit[['dataset', 'warnings']].drop_duplicates())


## Metrics and aggregation

Training and validation macro-F1 are measured separately, with gap = training minus validation F1. Depth counts edges from the root; nodes include internal and terminal nodes. Leaf occupancy counts training observations reaching each terminal leaf (unweighted), with its mean, median, fraction equal to one and fraction at most five. All leaves have positive training occupancy; the implementation checks counts against the fitted tree and training-set size.

Dataset metadata include filtered sample count, original predictor count (excluding target/IDs), number of classes, complete class counts, and imbalance = largest class count / smallest class count. Encoded predictor count is fold-dependent: its mean, sample SD, minimum and maximum are retained, rather than fitting an encoder on the full dataset.

First, every metric is summarized per dataset using the mean and sample SD (`ddof=1`) over 15 folds. These SDs describe variation in the repeated CV evaluations and are not uncertainty estimates from independent observations. Next, the equally weighted **dataset means** are summarized within each stratum using median, Q1, Q3, IQR and range. Folds are never treated as independent datasets. The mean of fold-specific leaf-occupancy medians is explicitly named `leaf_occupancy_median_mean`; leaves are not pooled across folds or datasets. No significance tests are performed.

**Observed preprocessing warnings:** Anneal has predictors with no observed training values; the original imputer drops them, with warnings captured per fold above and in `preprocessing_warnings.csv`. This is retained canonical behavior, not a new preprocessing choice. The transformed feature counts record the resulting widths.


In [ ]:
dataset_summary, stratum_summary, associations = summarize(context, fold_metrics, dataset_metadata)
pd.set_option('display.max_columns', 30)
display(dataset_summary[['dataset', 'stratum', 'n_samples', 'n_features_original',
    'n_features_transformed_mean', 'n_classes', 'imbalance_ratio', 'f1_train_mean',
    'f1_validation_mean', 'f1_validation_std', 'generalization_gap_mean',
    'depth_mean', 'leaves_mean', 'leaf_fraction_le_five_mean']])
display(stratum_summary.loc[stratum_summary.metric.isin([
    'f1_validation_mean', 'generalization_gap_mean', 'depth_mean', 'leaves_mean'])])


## Performance and structure by stratum

Colored points are individual dataset means. Black vertical segments span Q1–Q3; horizontal segments mark medians. Horizontal offsets separate points and have no numerical meaning. Labels identify extrema within strata. No representative tree is drawn, because these datasets have different predictors and targets.


In [ ]:
plot_comparisons(context, dataset_summary)


## Leaves, predictive performance and dataset size

Each point is one dataset. The sample-size axis is logarithmic; tree size and F1 remain on their original scales. Labels identify extrema to keep the figures legible. These relationships contextualize complexity; they do not adjust for confounders.


In [ ]:
plot_relations(context, dataset_summary)


## Training-leaf occupancy

The four panels retain both occupancy and small-leaf fractions. Points summarize folds within datasets; stratum summaries weight each dataset equally, irrespective of its sample count or number of leaves.


In [ ]:
plot_occupancy(context, dataset_summary)


## Context and interpretation

The following tables compare size, dimensionality, class count and imbalance and report descriptive Spearman rank associations across the 17 datasets. There are no p-values, significance claims or adjustments that would isolate causal effects. The accompanying prose is generated exclusively from the executed tables.


In [ ]:
display(stratum_summary.loc[stratum_summary.metric.isin([
    'n_samples', 'n_features_original', 'n_features_transformed_mean', 'n_classes', 'imbalance_ratio'])])
display(associations)
interpretation_md = interpretation(dataset_summary, stratum_summary, associations)
(context['out'] / 'interpretation.md').write_text(interpretation_md, encoding='utf-8')
display(Markdown(interpretation_md))


## Results generated from the current execution

The final cell generates the interpretation and manuscript paragraph directly from the current dataset and stratum summaries.


## Dependence on the canonical score and limits

Default-tree validation F1 already contributed to the canonical score alongside the balanced tree and two logistic-regression baselines. Consequently, poorer validation F1 in Hard can be partly built into the definition. This is a descriptive application, **not an independent validation** of characterization. Training gaps, leaf occupancies and structural metrics add information but do not establish that stratum membership causes it.

This experiment can organize behavioral comparisons and help ensure representation of different difficulty levels when choosing datasets. It does not compare random and difficulty-stratified selection, establish F1/efficiency gains, assess optimization or hyperheuristic benefits, or support claims about other models. No stratum guarantees a given tree size or degree of overfitting. The small groups, correlated repeated folds and differences in dataset size, dimension and class structure limit inference.

## English text for the paper

The paragraph below is exported to `results/default_tree_strata/paper_text_en.md`; its reported values are computed from the executed tables.


In [ ]:
paper_paragraph = paper_text(dataset_summary, stratum_summary)
(context['out'] / 'paper_text_en.md').write_text(paper_paragraph + '\n', encoding='utf-8')
display(Markdown(paper_paragraph))
print('Outputs:', context['out'])
print('Primary CSVs: fold_metrics.csv, dataset_metrics.csv, stratum_summary.csv')
print('Figures: performance_and_structure_by_stratum, leaf_performance_and_sample_size, training_leaf_occupancy_by_stratum (PNG/PDF)')
display(json.loads((context['out'] / 'validation_report.json').read_text()))


## Supporting implementation

The following tagged cell contains the helper implementation used by this supplement. The setup cell loads these definitions in a separate namespace, preserving the analysis variables and allowing reuse across notebooks.

In [ ]:
# Loaded by the setup cell; definitions are isolated from analysis variables.
if __name__ != '__main__':
    """Descriptive default-tree experiment using the read-only canonical artifacts.

    No model search, pooled training, or new difficulty labels are performed here.
    The companion notebook provides the narrative and executes these stages.
    """
    import ast
    import json
    import platform
    import warnings
    from pathlib import Path

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import scipy
    import sklearn
    from sklearn.metrics import f1_score
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.tree import DecisionTreeClassifier

    from experiments.characterization_persistence import (
        code_digest, data_digest, digest, validate_evaluations, write_json,
    )

    GROUPS = ['easy', 'intermediate', 'hard']
    COLORS = dict(easy='#009E73', intermediate='#0072B2', hard='#D55E00')
    METRICS = [
        'f1_train', 'f1_validation', 'generalization_gap', 'depth', 'nodes', 'leaves',
        'leaf_occupancy_mean', 'leaf_occupancy_median', 'leaf_fraction_one',
        'leaf_fraction_le_five', 'n_features_transformed',
    ]


    def canonical_components(root):
        """Reuse actual source definitions without running any canonical fits/exports."""
        path = root / 'experiments' / 'New_Characterization_Experiments.ipynb'
        notebook = json.loads(path.read_text(encoding='utf-8'))
        sources = [''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code']

        def cell_containing(marker):
            matches = [s for s in sources if marker in s]
            if len(matches) != 1:
                raise ValueError(f'Canonical source layout changed: {marker}')
            return matches[0]

        ns = {'__name__': 'canonical_read_only', 'display': lambda *args: None}
        # These three cells only import, locate inputs, and load CSVs into memory.
        for marker in ['# 0. Imports and setup', '# 1. Project root and datasets',
                       '# 2. Target configuration and loader']:
            exec(compile(cell_containing(marker), str(path), 'exec'), ns)
        baseline_source = cell_containing('def build_preprocessor(')
        boundary = '# Cache must be initialized before the canonical evaluation'
        if boundary not in baseline_source:
            raise ValueError('Cannot isolate read-only canonical baseline definitions')
        exec(compile(baseline_source.split(boundary)[0], str(path), 'exec'), ns)
        profile_ast = ast.parse(cell_containing('def assign_predictive_difficulty_groups('))
        definitions = ast.Module(body=[node for node in profile_ast.body
                                      if isinstance(node, ast.FunctionDef)], type_ignores=[])
        exec(compile(definitions, str(path), 'exec'), ns)
        if ns['PROJECT_ROOT'].resolve() != root.resolve():
            raise ValueError('Run from the project root or its experiments directory')
        return ns, path


    def prepare(root):
        """Fail before analysis on stale data, pool, labels, versions, or partitions."""
        root = Path(root).resolve()
        source = root / 'results' / 'new_characterization_experiments'
        out = root / 'results' / 'default_tree_strata'
        for folder in ['tables', 'figures']:
            (out / folder).mkdir(parents=True, exist_ok=True)
        write_json(out / 'validation_report.json', dict(complete=False, stage='preflight'))
        required = ['metadata.json', 'canonical_pool.json', 'canonical_model_configuration.json',
                    'tables/baseline_results.csv', 'tables/dataset_experimental_profile.csv',
                    'tables/ablation_long.csv', 'tables/cv_partitions.csv',
                    'tables/baseline_fold_scores.csv', 'tables/baseline_cv_audit.csv']
        missing = [str(source / p) for p in required if not (source / p).exists()]
        if missing:
            raise FileNotFoundError('Missing canonical artifacts; execute the canonical notebook first: '
                                    + ', '.join(missing))
        ns, notebook_path = canonical_components(root)
        metadata = json.loads((source / 'metadata.json').read_text())
        protocol = metadata['protocol']
        versions = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                        scipy=scipy.__version__, sklearn=sklearn.__version__)
        differences = {k: (protocol['versions'][k], v) for k, v in versions.items()
                       if protocol['versions'][k] != v}
        if differences:
            raise ValueError(f'Canonical/current version mismatch (no silent protocol change): {differences}')
        if protocol['cv'] != dict(folds=5, repeats=3, seed=42):
            raise ValueError('Canonical CV protocol changed')
        for name in ['load_dataset', '_resolve_column', '_resolve_existing_columns',
                     'build_preprocessor', 'make_baseline', 'evaluate_baselines']:
            if code_digest(ns[name]) != protocol['code'][name]:
                raise ValueError(f'Canonical definition differs from saved evaluation: {name}')
        for key, name in [('target_config', 'TARGET_CONFIG'), ('id_columns', 'ID_COLUMNS'),
                          ('missing_tokens', 'MISSING_TOKENS')]:
            if protocol[key] != ns[name]:
                raise ValueError(f'Canonical configuration changed: {key}')
        manifest = json.loads((source / 'canonical_pool.json').read_text())
        loaded = ns['loaded']
        if set(loaded) != set(manifest['selected_datasets']):
            raise ValueError('Selected dataset universe changed')
        for dataset, obj in loaded.items():
            identity = metadata['datasets'][dataset]
            if (digest(ns['datasets'][dataset].read_bytes()) != identity['file_sha256']
                    or data_digest(obj) != identity['filtered_data_sha256']):
                raise ValueError(f'{dataset}: canonical raw/filtered data identity mismatch')
        baselines = pd.read_csv(source / 'tables/baseline_results.csv', keep_default_na=False)
        for field in ['scores', 'tree_nodes', 'tree_depths', 'tree_leaves']:
            baselines[field] = baselines[field].map(json.loads)
        for field in ['f1_macro_mean', 'f1_macro_std', 'f1_macro_min', 'f1_macro_max',
                      'mean_tree_nodes', 'mean_tree_depth', 'mean_tree_leaves']:
            baselines[field] = pd.to_numeric(baselines[field], errors='coerce')
        audit = pd.read_csv(source / 'tables/baseline_cv_audit.csv')
        for row in audit.itertuples():
            if row.min_class_count != int(np.bincount(loaded[row.dataset]['y']).min()):
                raise ValueError(f'{row.dataset}: class-count audit changed')
        aggregated = ns['aggregate_baselines'](baselines, audit)
        eligible = aggregated.loc[aggregated.ranking_eligible, 'dataset'].tolist()
        if len(eligible) != 17 or set(eligible) != set(manifest['eligible']):
            raise ValueError(f'Pool discrepancy: expected saved 17 datasets, observed {eligible}')
        validate_evaluations(baselines, eligible, ns['NON_DUMMY_BASELINE_MODELS'])
        profile = pd.read_csv(source / 'tables/dataset_experimental_profile.csv')
        canonical = pd.read_csv(source / 'tables/ablation_long.csv')
        canonical = canonical.loc[canonical.variant.eq('canonical')].copy()
        if canonical.dataset.duplicated().any() or set(canonical.dataset) != set(eligible):
            raise ValueError('Canonical ranking has missing/duplicate datasets')
        # Reproduce the original calculation ONLY from saved original evaluations.
        aggregated['predictive_error'] = 1 - aggregated.baseline_macro_f1_mean
        norm = ns['minmax_normalize_on_pool']
        aggregated['predictive_difficulty_score'] = (
            .70 * norm(aggregated.predictive_error, aggregated.ranking_eligible)
            + .30 * norm(aggregated.cv_variability, aggregated.ranking_eligible))
        labels, ranks = ns['assign_predictive_difficulty_groups'](aggregated)
        aggregated['predictive_difficulty_label'] = labels
        aggregated['predictive_difficulty_rank'] = ranks
        reference = canonical.set_index('dataset').loc[eligible]
        for candidate in [aggregated, profile.loc[profile.ranking_eligible]]:
            if set(candidate.loc[candidate.ranking_eligible, 'dataset']) != set(eligible):
                raise ValueError('Canonical eligibility mismatch')
            candidate = candidate.set_index('dataset').loc[eligible]
            for target, original in [('difficulty_score', 'predictive_difficulty_score'),
                                     ('position', 'predictive_difficulty_rank'),
                                     ('F', 'baseline_macro_f1_mean'), ('V', 'cv_variability')]:
                np.testing.assert_allclose(reference[target], candidate[original], rtol=1e-12, atol=1e-14)
            if reference.difficulty_group.tolist() != candidate.predictive_difficulty_label.tolist():
                raise ValueError('Canonical label discrepancy: analysis stopped')
        if canonical.difficulty_group.value_counts().reindex(GROUPS).tolist() != [6, 6, 5]:
            raise ValueError('Canonical group counts changed')
        # Compare eligible historical labels, avoiding false mismatches for excluded rows.
        historical_path = root / 'results/dataset_statistics/dataset_experimental_profile.csv'
        history_note = 'Historical profile unavailable; only the specified canonical experiment is verified.'
        if historical_path.exists():
            historical = pd.read_csv(historical_path)
            hist_pool = historical.loc[historical.predictive_difficulty_label.isin(GROUPS)].copy()
            if set(hist_pool.dataset) != set(eligible) or hist_pool.dataset.duplicated().any():
                raise ValueError('Historical eligible pool differs; inspect before analysis')
            comparison = canonical[['dataset', 'difficulty_group', 'difficulty_score']].merge(
                hist_pool[['dataset', 'predictive_difficulty_label', 'predictive_difficulty_score']],
                on='dataset', validate='one_to_one')
            comparison.to_csv(out / 'tables/historical_label_audit.csv', index=False)
            if not comparison.difficulty_group.eq(comparison.predictive_difficulty_label).all():
                raise ValueError('Historical label discrepancy; see historical_label_audit.csv')
            max_delta = (comparison.difficulty_score - comparison.predictive_difficulty_score).abs().max()
            history_note = (f'All 17 historical labels agree. Maximum absolute score difference: {max_delta:.6g}. '
                            'Historical and current canonical environments differ; use the specified '
                            'New_Characterization_Experiments results, without claiming numerical paper reproduction.')
            print(history_note)
        parameters = DecisionTreeClassifier(random_state=42).get_params()
        saved_configs = json.loads((source / 'canonical_model_configuration.json').read_text())
        if parameters != DecisionTreeClassifier(**saved_configs['decision_tree_default']).get_params():
            raise ValueError('Current default tree differs from canonical baseline')
        partitions = pd.read_csv(source / 'tables/cv_partitions.csv')
        splits = {}
        for dataset in eligible:
            obj = loaded[dataset]
            rows = partitions.loc[partitions.dataset.eq(dataset)].sort_values(['repeat', 'fold'])
            keys = [(r, f) for r in range(1, 4) for f in range(1, 6)]
            if list(zip(rows.repeat, rows.fold)) != keys:
                raise ValueError(f'{dataset}: incomplete/duplicate canonical folds')
            expected = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42).split(obj['X'], obj['y'])
            splits[dataset] = []
            for row, (expected_train, expected_test) in zip(rows.itertuples(), expected):
                # Canonical hashes include the platform-specific index dtype (int32
                # on this Windows/NumPy version), not just the integer values.
                train = np.asarray(json.loads(row.train_indices), dtype=expected_train.dtype)
                test = np.asarray(json.loads(row.test_indices), dtype=expected_test.dtype)
                np.testing.assert_array_equal(train, expected_train)
                np.testing.assert_array_equal(test, expected_test)
                if (row.data_sha256 != data_digest(obj)
                        or row.split_sha256 != digest(train.tobytes() + test.tobytes())):
                    raise ValueError(f'{dataset}: split identity mismatch')
                splits[dataset].append((row.repeat, row.fold, train, test, row.split_sha256))
        exclusions = aggregated.loc[~aggregated.ranking_eligible,
                                   ['dataset', 'min_class_count', 'ranking_exclusion_reason']]
        exclusions.to_csv(out / 'tables/canonical_exclusions.csv', index=False)
        canonical.to_csv(out / 'tables/canonical_reference.csv', index=False)
        provenance = dict(versions=versions, tree_parameters=parameters, protocol=protocol['cv'],
                          canonical_history_note=history_note, std_ddof=1,
                          source_sha256={p: digest((source / p).read_bytes()) for p in required},
                          canonical_notebook_sha256=digest(notebook_path.read_bytes()),
                          implementation_sha256=digest((root / 'experiments' / 'Default_Tree_Canonical_Strata.ipynb').read_bytes()),
                          dataset_identities=metadata['datasets'], n_datasets=len(eligible),
                          scoring='macro_f1; zero_division=0',
                          imbalance_definition='maximum class count / minimum class count',
                          leaf_occupancy_definition='unweighted training observations reaching each terminal leaf')
        write_json(out / 'metadata.json', provenance)
        print(f'Preflight passed: {len(eligible)} datasets; Easy=6, Intermediate=6, Hard=5; 15 folds each.')
        return dict(root=root, source=source, out=out, ns=ns, canonical=canonical,
                    loaded=loaded, splits=splits, exclusions=exclusions, metadata=provenance)


    def fit_folds(context):
        """Fit fresh fold-local pipelines and check against original fold evaluations."""
        canonical = context['canonical'].sort_values('position')
        original = pd.read_csv(context['source'] / 'tables/baseline_fold_scores.csv')
        original = original.loc[original.model.eq('decision_tree_default')].set_index(
            ['dataset', 'repeat', 'fold'], verify_integrity=True)
        rows, info = [], []
        for reference in canonical.itertuples():
            dataset = reference.dataset
            obj = context['loaded'][dataset]
            X, y = obj['X'], obj['y']
            counts = obj['y_raw'].value_counts()
            info.append(dict(dataset=dataset, stratum=reference.difficulty_group,
                             canonical_score=reference.difficulty_score, canonical_rank=reference.position,
                             n_samples=len(y), n_features_original=X.shape[1], n_classes=len(counts),
                             imbalance_ratio=float(counts.max() / counts.min()),
                             class_distribution=json.dumps({str(k): int(v) for k, v in counts.items()}, sort_keys=True)))
            for repeat, fold, train, test, split_hash in context['splits'][dataset]:
                pipeline = Pipeline([
                    ('preprocess', context['ns']['build_preprocessor'](X, for_linear=False)),
                    ('model', DecisionTreeClassifier(random_state=42)),
                ])
                with warnings.catch_warnings(record=True) as captured:
                    warnings.simplefilter('always')
                    pipeline.fit(X.iloc[train], y[train])
                    train_f1 = f1_score(y[train], pipeline.predict(X.iloc[train]), average='macro', zero_division=0)
                    validation_f1 = f1_score(y[test], pipeline.predict(X.iloc[test]), average='macro', zero_division=0)
                    transformed = pipeline.named_steps['preprocess'].transform(X.iloc[train])
                tree = pipeline.named_steps['model']
                leaf_ids, occupancy = np.unique(tree.apply(transformed), return_counts=True)
                assert len(leaf_ids) == tree.get_n_leaves() and occupancy.sum() == len(train)
                np.testing.assert_array_equal(tree.tree_.n_node_samples[leaf_ids], occupancy)
                row = dict(dataset=dataset, stratum=reference.difficulty_group, repeat=repeat, fold=fold,
                           split_sha256=split_hash, data_sha256=data_digest(obj),
                           n_train=len(train), n_validation=len(test), n_features_transformed=transformed.shape[1],
                           f1_train=train_f1, f1_validation=validation_f1,
                           generalization_gap=train_f1-validation_f1, depth=tree.get_depth(),
                           nodes=tree.tree_.node_count, leaves=tree.get_n_leaves(),
                           leaf_occupancy_mean=float(occupancy.mean()),
                           leaf_occupancy_median=float(np.median(occupancy)),
                           leaf_fraction_one=float(np.mean(occupancy == 1)),
                           leaf_fraction_le_five=float(np.mean(occupancy <= 5)),
                           warnings='\n'.join(sorted({f'{w.category.__name__}: {w.message}' for w in captured})))
                saved = original.loc[(dataset, repeat, fold)]
                assert saved.split_sha256 == split_hash and saved.data_sha256 == row['data_sha256']
                np.testing.assert_allclose(validation_f1, saved.macro_f1, rtol=1e-12, atol=1e-14)
                for current, old in [('nodes', 'tree_nodes'), ('depth', 'tree_depths'), ('leaves', 'tree_leaves')]:
                    assert row[current] == saved[old], f'{dataset}/{repeat}/{fold}: {current} differs'
                rows.append(row)
            print(f'{dataset}: 15/15 folds match the canonical F1, nodes, depth and leaves.')
        folds = pd.DataFrame(rows)
        assert len(folds) == 255 and np.isfinite(folds[METRICS].to_numpy()).all()
        folds.to_csv(context['out'] / 'tables/fold_metrics.csv', index=False)
        return folds, pd.DataFrame(info)


    def summarize(context, folds, info):
        summary = folds.groupby('dataset')[METRICS].agg(['mean', 'std'])
        summary.columns = ['_'.join(c) for c in summary.columns]
        summary = info.merge(summary.reset_index(), on='dataset', validate='one_to_one')
        summary['n_folds'] = summary.dataset.map(folds.groupby('dataset').size())
        summary = summary.sort_values('canonical_rank').reset_index(drop=True)
        # Encoded width can differ by training fold; do not invent a global width.
        widths = folds.groupby('dataset').n_features_transformed.agg(['min', 'max'])
        for stat in ['min', 'max']:
            summary[f'n_features_transformed_{stat}'] = summary.dataset.map(widths[stat])
        columns = [m + '_mean' for m in METRICS] + [
            'n_samples', 'n_features_original', 'n_classes', 'imbalance_ratio', 'canonical_score']
        strata = []
        for group in GROUPS:
            subset = summary.loc[summary.stratum.eq(group)]
            for column in columns:
                values = subset[column]
                q1, median, q3 = values.quantile([.25, .5, .75])
                strata.append(dict(stratum=group, metric=column, n_datasets=len(subset),
                                   median=median, q1=q1, q3=q3, iqr=q3-q1,
                                   minimum=values.min(), maximum=values.max()))
        strata = pd.DataFrame(strata)
        # Descriptive rank associations only: no tests or p-values, n=17 datasets.
        associations = []
        for descriptor in ['n_samples', 'n_features_original', 'n_features_transformed_mean',
                           'n_classes', 'imbalance_ratio']:
            for metric in ['f1_validation_mean', 'generalization_gap_mean', 'depth_mean', 'leaves_mean']:
                rho = summary[descriptor].rank().corr(summary[metric].rank())
                associations.append(dict(descriptor=descriptor, metric=metric, n_datasets=len(summary),
                                         spearman_rho=rho))
        associations = pd.DataFrame(associations)
        for table, name in [(summary, 'dataset_metrics'), (strata, 'stratum_summary'),
                            (associations, 'descriptive_associations')]:
            table.to_csv(context['out'] / f'tables/{name}.csv', index=False)
        write_json(context['out'] / 'validation_report.json', dict(
            n_datasets=len(summary), n_folds=len(folds), all_canonical_folds_match=True,
            canonical_data_and_code_verified=True, saved_partitions_verified=True,
            dataset_counts=summary.stratum.value_counts().to_dict(),
            folds_with_warnings=int(folds.warnings.ne('').sum()),
            sample_std_ddof=1, analysis_unit='dataset', complete=True))
        return summary, strata, associations


    def save_figure(context, fig, name):
        fig.tight_layout()
        for suffix in ['png', 'pdf']:
            fig.savefig(context['out'] / f'figures/{name}.{suffix}', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close(fig)


    def group_panel(ax, summary, metric, label, annotate=False):
        for i, group in enumerate(GROUPS):
            subset = summary.loc[summary.stratum.eq(group)].sort_values(metric)
            values = subset[metric]
            offsets = np.linspace(-.18, .18, len(values))
            ax.scatter(i + offsets, values, color=COLORS[group], s=38, alpha=.85, zorder=3)
            q1, median, q3 = values.quantile([.25, .5, .75])
            ax.plot([i, i], [q1, q3], color='black', linewidth=3, zorder=2)
            ax.plot([i-.09, i+.09], [median, median], color='black', linewidth=2, zorder=2)
            if annotate:
                for j in sorted({0, len(subset)-1}):
                    ax.annotate(subset.iloc[j].dataset, (i+offsets[j], values.iloc[j]),
                                xytext=(4, -12 if j == 0 else 5), textcoords='offset points', fontsize=7)
        ax.set(xticks=range(3), xticklabels=[f'{g.title()}\n(n={sum(summary.stratum.eq(g))})' for g in GROUPS],
               ylabel=label, xlim=(-.5, 2.65))
        ax.margins(y=.18)
        ax.grid(axis='y', alpha=.2)


    def plot_comparisons(context, summary):
        fig, axes = plt.subplots(2, 2, figsize=(13, 9))
        for ax, metric, label in zip(axes.flat,
                ['f1_validation_mean', 'generalization_gap_mean', 'depth_mean', 'leaves_mean'],
                ['Validation macro-F1', 'Train minus validation macro-F1', 'Tree depth', 'Number of leaves']):
            group_panel(ax, summary, metric, label, annotate=True)
        fig.suptitle('Default trees by canonical difficulty\nDots: dataset CV means; black bars: median and interquartile range')
        save_figure(context, fig, 'performance_and_structure_by_stratum')


    def plot_relations(context, summary):
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        pairs = [('leaves_mean', 'f1_validation_mean', 'Mean number of leaves', 'Mean validation macro-F1'),
                 ('n_samples', 'leaves_mean', 'Dataset observations (log scale)', 'Mean number of leaves')]
        for ax, (x, y, xlabel, ylabel) in zip(axes, pairs):
            for group in GROUPS:
                s = summary.loc[summary.stratum.eq(group)]
                ax.scatter(s[x], s[y], color=COLORS[group], label=group.title(), s=48)
            if x == 'n_samples':
                ax.set_xscale('log')
            # Label extrema to aid interpretation without crowding all 17 points.
            chosen = {summary[x].idxmin(), summary[x].idxmax(), summary[y].idxmin(), summary[y].idxmax()}
            for idx in sorted(chosen):
                row = summary.loc[idx]
                ax.annotate(row.dataset, (row[x], row[y]), xytext=(5, 6),
                            textcoords='offset points', fontsize=8)
            ax.set(xlabel=xlabel, ylabel=ylabel)
            ax.margins(x=.25, y=.18)
            ax.grid(alpha=.2)
        axes[0].legend()
        fig.suptitle('Dataset-level relationships (one point per dataset)')
        save_figure(context, fig, 'leaf_performance_and_sample_size')


    def plot_occupancy(context, summary):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        for ax, metric, label in zip(axes.flat,
                ['leaf_occupancy_mean_mean', 'leaf_occupancy_median_mean',
                 'leaf_fraction_one_mean', 'leaf_fraction_le_five_mean'],
                ['Mean training observations per leaf', 'Median training observations per leaf',
                 'Fraction of leaves with 1 observation', 'Fraction of leaves with <=5 observations']):
            group_panel(ax, summary, metric, label)
        fig.suptitle('Training leaf occupancy\nDots: dataset CV means; black bars: median and interquartile range')
        save_figure(context, fig, 'training_leaf_occupancy_by_stratum')


    def interpretation(summary, strata, associations):
        def fmt(group, metric):
            r = strata.loc[strata.stratum.eq(group) & strata.metric.eq(metric)].iloc[0]
            return f'{r["median"]:.3f} [{r.q1:.3f}, {r.q3:.3f}]'

        paragraphs = ['## Observed results\n\nAll values below are computed from the executed folds. '
                      'Bracketed values are the 25th and 75th percentiles across dataset means; '
                      'they are descriptive intervals, not confidence intervals.']
        for group in GROUPS:
            paragraphs.append(f'**{group.title()} (n={sum(summary.stratum.eq(group))}).** '
                f'Median validation macro-F1: {fmt(group, "f1_validation_mean")}; '
                f'gap: {fmt(group, "generalization_gap_mean")}; '
                f'depth: {fmt(group, "depth_mean")}; leaves: {fmt(group, "leaves_mean")}; '
                f'fraction of leaves with at most five observations: {fmt(group, "leaf_fraction_le_five_mean")}.')
        for metric, label in [('f1_validation_mean', 'Validation F1'),
                               ('generalization_gap_mean', 'Generalization gap'),
                               ('depth_mean', 'Depth'), ('leaves_mean', 'Leaf count')]:
            pieces = []
            for a, b in [('easy', 'intermediate'), ('intermediate', 'hard'), ('easy', 'hard')]:
                av = summary.loc[summary.stratum.eq(a), metric]
                bv = summary.loc[summary.stratum.eq(b), metric]
                lower, upper = max(av.min(), bv.min()), min(av.max(), bv.max())
                pieces.append(f'{a}/{b}: ' + (f'overlapping ranges [{lower:.3f}, {upper:.3f}]'
                                             if lower <= upper else 'non-overlapping observed ranges'))
            paragraphs.append(f'**{label} overlap.** ' + '; '.join(pieces) + '. '
                              'Range overlap describes this small collection only.')
        largest = summary.loc[summary.leaves_mean.idxmax()]
        smallest = summary.loc[summary.leaves_mean.idxmin()]
        best_hard = summary.loc[summary.stratum.eq('hard')].sort_values('f1_validation_mean').iloc[-1]
        worst_easy = summary.loc[summary.stratum.eq('easy')].sort_values('f1_validation_mean').iloc[0]
        paragraphs.append(f'**Heterogeneity and exceptions.** The largest mean tree is {largest.dataset} '
            f'({largest.stratum}, {largest.leaves_mean:.1f} leaves, n={largest.n_samples}); the smallest is '
            f'{smallest.dataset} ({smallest.stratum}, {smallest.leaves_mean:.1f} leaves, n={smallest.n_samples}). '
            f'The highest default-tree F1 within Hard is {best_hard.dataset} ({best_hard.f1_validation_mean:.3f}); '
            f'the lowest within Easy is {worst_easy.dataset} ({worst_easy.f1_validation_mean:.3f}). '
            'Canonical labels describe a four-model composite and do not impose a strict ordering on every tree descriptor.')
        larger_hard = int((summary.loc[summary.stratum.eq('hard'), 'leaves_mean'] < worst_easy.leaves_mean).sum())
        smaller_easy = int((summary.loc[summary.stratum.eq('easy'), 'leaves_mean'] > best_hard.leaves_mean).sum())
        paragraphs.append(f'{worst_easy.dataset}, despite its Easy label, averages {worst_easy.leaves_mean:.1f} '
            f'leaves, exceeding {larger_hard} of the 5 Hard datasets, with a gap of '
            f'{worst_easy.generalization_gap_mean:.3f}. Conversely, {best_hard.dataset} (Hard) averages '
            f'{best_hard.leaves_mean:.1f} leaves, fewer than {smaller_easy} of the 6 Easy datasets. '
            'Thus the stratum medians do not imply uniformly ordered tree sizes. '
            f'Intermediate and Hard median depths are {fmt("intermediate", "depth_mean")} and '
            f'{fmt("hard", "depth_mean")}, respectively; their central depths are very similar in this collection.')
        perfect = int(np.isclose(summary.f1_train_mean, 1.0, rtol=0, atol=1e-12).sum())
        paragraphs.append(f'**Training fit.** {perfect} of {len(summary)} datasets have mean training '
            'macro-F1 equal to 1.0. For these datasets, the gap is exactly one minus validation F1, '
            'so its ordering adds no independent ranking information. The remaining training-fit '
            'values and the leaf-occupancy metrics help distinguish the observed profiles; '
            'they do not by themselves establish why training or validation errors occur.')
        paragraphs.append('**Metadata across strata.** In Easy, Intermediate, Hard order, median sample '
            'counts [Q1, Q3] are ' + '; '.join(fmt(g, 'n_samples') for g in GROUPS) +
            '; original predictor counts are ' + '; '.join(fmt(g, 'n_features_original') for g in GROUPS) +
            '; class counts are ' + '; '.join(fmt(g, 'n_classes') for g in GROUPS) +
            '; and imbalance ratios are ' + '; '.join(fmt(g, 'imbalance_ratio') for g in GROUPS) + '. '
            'These summaries and the individual points reveal that the groups differ in task composition '
            'as well as predictive difficulty.')
        associations_text = []
        for descriptor in ['n_samples', 'n_features_original', 'n_features_transformed_mean', 'n_classes', 'imbalance_ratio']:
            s = associations.loc[associations.descriptor.eq(descriptor)].set_index('metric')
            associations_text.append(f'{descriptor}: leaf count rho={s.loc["leaves_mean", "spearman_rho"]:.3f}, '
                                     f'gap rho={s.loc["generalization_gap_mean", "spearman_rho"]:.3f}')
        paragraphs.append('**Dataset context.** Descriptive Spearman associations across 17 datasets are: '
            + '; '.join(associations_text) + '. These are unadjusted rank associations, without significance tests. '
            'The metadata summary reports size, original and encoded dimensionality, class count and max/min '
            'class-frequency imbalance by stratum. These descriptors co-vary; the experiment cannot isolate '
            'a difficulty effect from size, dimensionality or class structure. Leaf counts alone are not a '
            'size-adjusted complexity comparison.')
        paragraphs.append('**Added information and scope.** Gap and leaf occupancy describe training fit and '
            'fragmentation beyond the canonical validation score; neither proves that difficulty causes a '
            'particular structure. The same default-tree validation scores contribute to the canonical '
            'four-model score (70% normalized predictive error, 30% normalized within-model CV variability). '
            'Lower F1 in Hard is therefore partly incorporated in the definition, and this is not independent '
            'validation of the characterization. The grouping organizes observed profiles and can be used '
            'to control difficulty representation when selecting datasets. No random-versus-stratified '
            'sampling comparison, F1 or efficiency benefit, hyperparameter-optimization or hyperheuristic '
            'benefit, or generalization to other models has been tested. A stratum guarantees neither tree '
            'size nor overfitting. Difficulty stratification selects datasets; class stratification distributes '
            'labels within CV folds. These are distinct operations.')
        return '\n\n'.join(paragraphs)


    def paper_text(summary, strata):
        lookup = strata.set_index(['stratum', 'metric'])
        f1 = ', '.join(f'{lookup.loc[(g, "f1_validation_mean"), "median"]:.3f}' for g in GROUPS)
        gaps = ', '.join(f'{lookup.loc[(g, "generalization_gap_mean"), "median"]:.3f}' for g in GROUPS)
        leaves = ', '.join(f'{lookup.loc[(g, "leaves_mean"), "median"]:.1f}' for g in GROUPS)
        return ('We used the canonical difficulty strata to organize a descriptive evaluation of default '
            'decision trees on the same 17 eligible datasets (6 Easy, 6 Intermediate, and 5 Hard). '
            'A separate tree was fitted in every fold using the original repeated stratified five-fold '
            'protocol (three repetitions), saved partitions, and training-only preprocessing. '
            'We first averaged the 15 evaluations within each dataset, then summarized these dataset-level '
            'means with equal dataset weight. Median validation macro-F1 was ' + f1 +
            ', and median train-minus-validation F1 was ' + gaps +
            ' for Easy, Intermediate, and Hard, respectively. Median leaf counts were ' + leaves +
            '. Tree structure and training-leaf occupancy provided complementary descriptions of '
            'heterogeneity, while differences in dataset size, dimensionality and class structure limit '
            'attribution to difficulty alone. This application illustrates how the strata can organize '
            'behavioral profiles and control difficulty representation in benchmark selection. Because '
            'the default-tree validation results also contributed to the canonical difficulty score, '
            'the comparison is not an independent validation of the characterization. We did not evaluate '
            'whether difficulty-stratified dataset selection improves predictive accuracy or efficiency '
            'over random selection, benefits hyperparameter optimization or a hyperheuristic, or '
            'generalizes to other models. Stratum membership does not guarantee tree size or overfitting. '
            'Dataset stratification by difficulty is distinct from class stratification within CV.')
